In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.491, 10: 0.503, 20: 0.5115000000000001, 30: 0.5115000000000001, 40: 0.542, 50: 0.5499999999999999, 60: 0.5515000000000001, 70: 0.5614999999999999, 80: 0.5765, 90: 0.6019999999999999, 100: 0.6125, 110: 0.6230000000000001, 120: 0.6459999999999997, 130: 0.6489999999999999, 140: 0.6684999999999999, 150: 0.6844999999999999, 160: 0.7060000000000001, 170: 0.7230000000000001, 180: 0.7310000000000001, 190: 0.7479999999999999, 200: 0.748, 210: 0.7699999999999999, 220: 0.7665, 230: 0.7855, 240: 0.7925000000000001, 250: 0.8050000000000003, 260: 0.8080000000000002, 270: 0.8125, 280: 0.8125, 290: 0.8175000000000001, 300: 0.8240000000000004}
{0: 0.004719, 10: 0.005491000000000001, 20: 0.005237750000000001, 30: 0.004097750000000001, 40: 0.0062759999999999995, 50: 0.00562, 60: 0.00437775, 70: 0.006867749999999999, 80: 0.0066577500000000005, 90: 0.0043760000000000005, 100: 0.0041737499999999995, 110: 0.006171, 120: 0.005604, 130: 0.007239, 140: 0.006737749999999999, 150: 0.006749749999999998, 160: